# Train the hazard detector
Set the runtime to a GPU first: Runtime > Change runtime type > T4 GPU. Then run the cells in order.
Do not paste API keys into cells. If you use Roboflow, type the key into the prompt when asked.

In [ ]:
!pip -q install ultralytics onnxruntime

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())

## 1. Dataset
Pick one option. The dataset must be in YOLOv8 format and its licence must allow your use (see docs/DATA.md).

In [ ]:
# Option A: upload a dataset zip from your computer
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
!unzip -q -o "{zip_name}" -d /content/dataset

In [ ]:
# Option B: Roboflow export. Remove the leading # on each line to use it.
# !pip -q install roboflow
# from getpass import getpass
# from roboflow import Roboflow
# rf = Roboflow(api_key=getpass('Roboflow API key: '))
# project = rf.workspace('WORKSPACE').project('PROJECT')
# project.version(1).download('yolov8', location='/content/dataset')

In [ ]:
!ls /content/dataset
!cat /content/dataset/data.yaml

## 2. Train, check, export
If Colab disconnects, lower `epochs` or mount Drive and save runs there.

In [ ]:
!yolo detect train data=/content/dataset/data.yaml model=yolov8n.pt imgsz=640 epochs=60 batch=16 project=/content/runs name=odr

In [ ]:
!yolo detect val model=/content/runs/odr/weights/best.pt data=/content/dataset/data.yaml

In [ ]:
!yolo export model=/content/runs/odr/weights/best.pt format=onnx imgsz=640 opset=12 simplify=True
!find /content/runs -name best.onnx

## 3. Check the export and get the labels line

In [ ]:
from ultralytics import YOLO
import onnxruntime as ort, numpy as np
m = YOLO('/content/runs/odr/weights/best.pt')
s = ort.InferenceSession('/content/runs/odr/weights/best.onnx', providers=['CPUExecutionProvider'])
out = s.run(None, {s.get_inputs()[0].name: np.zeros((1, 3, 640, 640), dtype=np.float32)})[0]
print('output shape:', out.shape, '-> classes:', out.shape[1] - 4)
print('Paste this into src/labels.ts:')
print('export const LABELS = ' + str(list(m.names.values())) + ';')

In [ ]:
import shutil
from google.colab import files
shutil.copy('/content/runs/odr/weights/best.onnx', '/content/hazard.onnx')
files.download('/content/hazard.onnx')

Put the downloaded file at `public/models/hazard.onnx`, replace `src/labels.ts` with the printed line, and copy the mAP numbers from the validation output into the README model card.